# 🧬 Molecular Precision Medicine — Task 3
## Molecular Patient Subtyping from Bulk RNA-seq Data

**Domain:** Molecular & Omics Based Personalized Medicine  
**Subdomain:** Molecular Patient Subtyping  
**Time Limit:** 6 hours


---
## 📋 Task Prompt

### Background
You are given bulk RNA-seq gene expression data and paired clinical metadata for a cohort of breast cancer patients. Each patient has been profiled across 500 genes. Your task is to build a computational pipeline that performs **unsupervised molecular subtyping**, followed by **supervised validation** and **differential expression analysis** — reproducing a standard workflow in precision oncology.

---

### Objective
Implement a complete molecular subtyping pipeline that:

1. **Selects Highly Variable Genes (HVG):** Identify the top 100 genes by expression variance across patients.
2. **Performs PCA:** Apply PCA on the HVG-scaled expression matrix (StandardScaler → PCA, 15 components). Report how many PCs are needed to explain ≥90% cumulative variance, and the variance explained by PC1.
3. **Clusters Patients:** Apply K-Means (k=3, random_state=42, n_init=20) on the first 10 PCs. Report the **Silhouette Score** and **Adjusted Rand Index (ARI)** against the ground-truth `molecular_subtype` labels.
4. **Trains a Subtype Classifier:** Train a Random Forest classifier (n_estimators=200, random_state=42) using `SelectKBest` (f_classif, k=50) for feature selection on the HVG matrix. Report **5-fold stratified cross-validation Balanced Accuracy** (mean ± std).
5. **Identifies Differentially Expressed Genes:** Using the cluster assignments from step 3, apply the Kruskal-Wallis test (scipy.stats.kruskal) per gene across the 3 clusters on the HVG matrix. Report the **top 10 DE genes** by ascending p-value.
6. **Generates a Subtype Clinical Summary:** For each cluster report: patient count, mean age, ER-positive percentage, median OS months, and event rate.

---

### Input Format
Two CSV files:

**`{prefix}_expression.csv`**
```
patient_id, GENE_0000, GENE_0001, ..., GENE_0499
P001,       1.23,      -0.45,     ..., 0.78
...
```

**`{prefix}_clinical.csv`**
```
patient_id, age, stage, er_status, her2_status, molecular_subtype, os_months, event
P001,       55,  II,    Positive,  Negative,    Luminal_A,          48,       0
...
```

---

### Output Format
Your solution must print or return a dictionary with **exactly** these keys:

| Key | Type | Description |
|-----|------|-------------|
| `n_patients` | int | Number of patients loaded |
| `n_genes` | int | Total genes in expression file |
| `n_hvg` | int | Number of HVGs selected (should be 100) |
| `pcs_explaining_90pct_variance` | int | Minimum PCs for ≥90% cumulative variance |
| `pc1_variance_explained` | float | Fraction of variance explained by PC1 (4 decimals) |
| `silhouette_score` | float | Silhouette score of K-Means clustering (4 decimals) |
| `adjusted_rand_index` | float | ARI vs ground truth subtypes (4 decimals) |
| `rf_cv_balanced_accuracy_mean` | float | Mean 5-CV balanced accuracy (4 decimals) |
| `rf_cv_balanced_accuracy_std` | float | Std 5-CV balanced accuracy (4 decimals) |
| `top_de_genes` | list[str] | Top 10 DE gene names by p-value (ascending) |
| `subtype_summary` | dict | Output of per-cluster clinical summary |

---

### Example
**Input files:** `example_expression.csv`, `example_clinical.csv` (120 patients, 500 genes)

**Expected output (key metrics):**
```json
{
  "n_patients": 120,
  "n_genes": 500,
  "n_hvg": 100,
  "pcs_explaining_90pct_variance": 7,
  "pc1_variance_explained": 0.8677,
  "silhouette_score": 0.549,
  "adjusted_rand_index": 1.0,
  "rf_cv_balanced_accuracy_mean": 1.0,
  "rf_cv_balanced_accuracy_std": 0.0,
  "top_de_genes": ["GENE_0012","GENE_0032","GENE_0016","GENE_0021","GENE_0020",
                   "GENE_0011","GENE_0029","GENE_0046","GENE_0031","GENE_0033"]
}
```

---

### Constraints
- Use **scikit-learn ≥ 1.2**, **scipy ≥ 1.10**, **pandas**, **numpy**
- Do **not** use pre-trained models or external gene databases
- All random seeds must be set as specified; do not change them
- Gene column names follow the pattern `GENE_XXXX`


---
## ⚙️ Setup & Imports

In [ ]:
# Install dependencies
!pip install scikit-learn scipy pandas numpy --quiet

In [ ]:
import pandas as pd
import numpy as np
import json
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.feature_selection import SelectKBest, f_classif
from scipy.stats import kruskal

print("All imports successful ✓")

---
## 📂 Load Example Data

Mount your Google Drive or upload files and update paths below.

In [ ]:
# ── Update these paths ──────────────────────────────────────────────────────
EXPRESSION_CSV = 'example_data/example_expression.csv'
CLINICAL_CSV   = 'example_data/example_clinical.csv'
# ─────────────────────────────────────────────────────────────────────────────

expr    = pd.read_csv(EXPRESSION_CSV)
clin    = pd.read_csv(CLINICAL_CSV)
merged  = clin.merge(expr, on='patient_id')
gene_cols = [c for c in expr.columns if c.startswith('GENE_')]

print(f"Patients  : {len(merged)}")
print(f"Genes     : {len(gene_cols)}")
print(f"Subtypes  : {merged['molecular_subtype'].value_counts().to_dict()}")
merged.head(3)

---
## Step 1 — Highly Variable Gene Selection

In [ ]:
expr_matrix = merged[gene_cols]

# Top 100 genes by variance
variances = expr_matrix.var(axis=0)
hvg = variances.nlargest(100).index.tolist()
expr_hvg = expr_matrix[hvg]

print(f"Selected {len(hvg)} HVGs")
print(f"Top 5 HVGs by variance: {hvg[:5]}")

---
## Step 2 — PCA

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(expr_hvg)

pca = PCA(n_components=15, random_state=42)
pcs = pca.fit_transform(X_scaled)

explained = pca.explained_variance_ratio_
cumvar = np.cumsum(explained)
n90 = int(np.searchsorted(cumvar, 0.90)) + 1

print(f"Cumulative variance (first 10 PCs): {np.round(cumvar[:10], 3)}")
print(f"PCs needed for ≥90% variance      : {n90}")
print(f"PC1 variance explained             : {explained[0]:.4f}")

---
## Step 3 — K-Means Clustering

In [ ]:
true_labels = merged['molecular_subtype'].values

km = KMeans(n_clusters=3, random_state=42, n_init=20)
pred_labels = km.fit_predict(pcs[:, :10])
merged['cluster'] = pred_labels

sil  = silhouette_score(pcs[:, :10], pred_labels)
ari  = adjusted_rand_score(true_labels, pred_labels)

print(f"Silhouette Score          : {sil:.4f}")
print(f"Adjusted Rand Index (ARI) : {ari:.4f}")

---
## Step 4 — Random Forest Subtype Classifier

In [ ]:
selector = SelectKBest(f_classif, k=50)
X_sel    = selector.fit_transform(expr_hvg.values, true_labels)

clf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
cv  = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(clf, X_sel, true_labels, cv=cv, scoring='balanced_accuracy')

clf.fit(X_sel, true_labels)

print(f"5-CV Balanced Accuracy : {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
print(f"Per-fold scores        : {np.round(cv_scores, 4)}")

---
## Step 5 — Differential Expression (Kruskal-Wallis)

In [ ]:
de_results = []
expr_hvg_reset = expr_hvg.reset_index(drop=True)

for gene in hvg:
    groups = [expr_hvg_reset[pred_labels == lbl][gene].values
              for lbl in np.unique(pred_labels)]
    stat, pval = kruskal(*groups)
    de_results.append({'gene': gene, 'statistic': stat, 'pval': pval})

de_df = pd.DataFrame(de_results).sort_values('pval').head(10)
top_de_genes = de_df['gene'].tolist()

print("Top 10 Differentially Expressed Genes:")
print(de_df[['gene', 'pval']].to_string(index=False))

---
## Step 6 — Per-Subtype Clinical Summary

In [ ]:
summary = merged.groupby('cluster').agg(
    n_patients   = ('patient_id',   'count'),
    mean_age     = ('age',          'mean'),
    er_positive_pct = ('er_status', lambda x: (x=='Positive').mean()*100),
    median_os_months = ('os_months','median'),
    event_rate   = ('event',        'mean')
).round(2)

print(summary.to_string())

---
## 📊 Final Results Dictionary

In [ ]:
results = {
    'n_patients'                     : int(len(merged)),
    'n_genes'                        : int(len(gene_cols)),
    'n_hvg'                          : int(len(hvg)),
    'pcs_explaining_90pct_variance'  : int(n90),
    'pc1_variance_explained'         : round(float(explained[0]), 4),
    'silhouette_score'               : round(float(sil), 4),
    'adjusted_rand_index'            : round(float(ari), 4),
    'rf_cv_balanced_accuracy_mean'   : round(float(cv_scores.mean()), 4),
    'rf_cv_balanced_accuracy_std'    : round(float(cv_scores.std()),  4),
    'top_de_genes'                   : top_de_genes,
    'subtype_summary'                : summary.to_dict()
}

print(json.dumps({k: v for k, v in results.items() if k != 'subtype_summary'}, indent=2))

---
## 🧪 Test Cases

The following 5 test cases evaluate an LLM solution against held-out datasets.  
Test data files are in `test_data/` and are **not** referenced in the task prompt.


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# TEST CASE RUNNER
# Evaluates LLM outputs against ground-truth expected values
# ══════════════════════════════════════════════════════════════════════════════

TOLERANCE = 0.05  # Allowed absolute deviation for float metrics

GROUND_TRUTH = {
    "test1": {
        "n_patients": 100,
        "n_genes": 500,
        "n_hvg": 100,
        "pcs_explaining_90pct_variance": 7,
        "pc1_variance_explained": 0.8661,
        "silhouette_score": 0.5327,
        "adjusted_rand_index": 1.0,
        "rf_cv_balanced_accuracy_mean": 1.0,
        "top_de_genes_set": {"GENE_0031","GENE_0006","GENE_0000","GENE_0014","GENE_0022",
                             "GENE_0032","GENE_0008","GENE_0045","GENE_0048","GENE_0044"}
    },
    "test2": {
        "n_patients": 150,
        "n_genes": 500,
        "n_hvg": 100,
        "pcs_explaining_90pct_variance": 8,
        "pc1_variance_explained": 0.8666,
        "silhouette_score": 0.5672,
        "adjusted_rand_index": 1.0,
        "rf_cv_balanced_accuracy_mean": 1.0,
        "top_de_genes_set": {"GENE_0040","GENE_0015","GENE_0009","GENE_0011","GENE_0039",
                             "GENE_0035","GENE_0038","GENE_0047","GENE_0049","GENE_0008"}
    },
    "test3": {
        "n_patients": 90,
        "n_genes": 500,
        "n_hvg": 100,
        "pcs_explaining_90pct_variance": 7,
        "pc1_variance_explained": 0.8623,
        "silhouette_score": 0.537,
        "adjusted_rand_index": 1.0,
        "rf_cv_balanced_accuracy_mean": 1.0,
        "top_de_genes_set": {"GENE_0046","GENE_0029","GENE_0005","GENE_0020","GENE_0025",
                             "GENE_0018","GENE_0037","GENE_0009","GENE_0012","GENE_0034"}
    }
}


def evaluate_result(result, gt, dataset_name):
    passed, failed = [], []

    def check(name, pred, expected, is_float=False, is_set=False):
        if is_set:
            overlap = len(set(pred) & expected)
            ok = overlap >= 8  # at least 8/10 genes must match
            status = f"PASS (overlap={overlap}/10)" if ok else f"FAIL (overlap={overlap}/10)"
        elif is_float:
            ok = abs(pred - expected) <= TOLERANCE
            status = f"PASS ({pred})" if ok else f"FAIL (got {pred}, expected {expected}±{TOLERANCE})"
        else:
            ok = pred == expected
            status = f"PASS ({pred})" if ok else f"FAIL (got {pred}, expected {expected})"
        (passed if ok else failed).append(f"  {name}: {status}")

    print(f"\n{'─'*60}")
    print(f"Dataset: {dataset_name}")
    print(f"{'─'*60}")

    check("TC1 — n_patients",                result['n_patients'],                    gt['n_patients'])
    check("TC2 — n_hvg = 100",               result['n_hvg'],                         gt['n_hvg'])
    check("TC3 — pcs_for_90pct_variance",    result['pcs_explaining_90pct_variance'], gt['pcs_explaining_90pct_variance'])
    check("TC4 — silhouette_score",          result['silhouette_score'],              gt['silhouette_score'],     is_float=True)
    check("TC5 — adjusted_rand_index ≥ 0.9", result['adjusted_rand_index'],           gt['adjusted_rand_index'],  is_float=True)
    check("TC6 — rf_balanced_accuracy ≥ 0.9",result['rf_cv_balanced_accuracy_mean'],  gt['rf_cv_balanced_accuracy_mean'], is_float=True)
    check("TC7 — top_de_genes (≥8/10 match)",result['top_de_genes'],                  gt['top_de_genes_set'],     is_set=True)

    all_checks = passed + failed
    for line in all_checks:
        icon = "✅" if "PASS" in line else "❌"
        print(f"{icon} {line.strip()}")

    n_pass = len(passed)
    n_total = len(all_checks)
    print(f"\nScore: {n_pass}/{n_total} checks passed")
    return n_pass, n_total


# ── Run tests ────────────────────────────────────────────────────────────────
# Import the pipeline function (assumes expert_solution.py is in the same dir,
# OR replace with your own pipeline call)

import importlib, sys
sys.path.insert(0, '.')
# from expert_solution import run_pipeline  # uncomment if running as module

# For demonstration, re-run inline pipeline on each test set:
def run_pipeline_inline(expression_csv, clinical_csv):
    import pandas as pd, numpy as np
    from sklearn.preprocessing import StandardScaler
    from sklearn.decomposition import PCA
    from sklearn.cluster import KMeans
    from sklearn.metrics import adjusted_rand_score, silhouette_score
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.model_selection import StratifiedKFold, cross_val_score
    from sklearn.feature_selection import SelectKBest, f_classif
    from scipy.stats import kruskal
    import warnings; warnings.filterwarnings('ignore')

    expr   = pd.read_csv(expression_csv)
    clin   = pd.read_csv(clinical_csv)
    merged = clin.merge(expr, on='patient_id')
    gene_cols = [c for c in expr.columns if c.startswith('GENE_')]
    expr_matrix = merged[gene_cols]
    true_labels = merged['molecular_subtype'].values

    hvg = expr_matrix.var(axis=0).nlargest(100).index.tolist()
    expr_hvg = expr_matrix[hvg]

    X_scaled = StandardScaler().fit_transform(expr_hvg)
    pca = PCA(n_components=15, random_state=42)
    pcs = pca.fit_transform(X_scaled)
    explained = pca.explained_variance_ratio_
    cumvar = np.cumsum(explained)
    n90 = int(np.searchsorted(cumvar, 0.90)) + 1

    km = KMeans(n_clusters=3, random_state=42, n_init=20)
    pred_labels = km.fit_predict(pcs[:,:10])
    sil = silhouette_score(pcs[:,:10], pred_labels)
    ari = adjusted_rand_score(true_labels, pred_labels)

    X_sel = SelectKBest(f_classif, k=50).fit_transform(expr_hvg.values, true_labels)
    clf   = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
    cv    = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    cv_sc = cross_val_score(clf, X_sel, true_labels, cv=cv, scoring='balanced_accuracy')

    de_res = []
    ehvg = expr_hvg.reset_index(drop=True)
    for g in hvg:
        gs = [ehvg[pred_labels==l][g].values for l in np.unique(pred_labels)]
        _, pv = kruskal(*gs)
        de_res.append({'gene':g,'pval':pv})
    top10 = pd.DataFrame(de_res).sort_values('pval').head(10)['gene'].tolist()

    return {
        'n_patients': len(merged), 'n_genes': len(gene_cols), 'n_hvg': len(hvg),
        'pcs_explaining_90pct_variance': n90,
        'pc1_variance_explained': round(float(explained[0]),4),
        'silhouette_score': round(float(sil),4),
        'adjusted_rand_index': round(float(ari),4),
        'rf_cv_balanced_accuracy_mean': round(float(cv_sc.mean()),4),
        'rf_cv_balanced_accuracy_std':  round(float(cv_sc.std()),4),
        'top_de_genes': top10
    }

total_pass = total_checks = 0
for ts in ['test1','test2','test3']:
    r = run_pipeline_inline(
        f'test_data/{ts}_expression.csv',
        f'test_data/{ts}_clinical.csv'
    )
    p, t = evaluate_result(r, GROUND_TRUTH[ts], ts)
    total_pass += p; total_checks += t

print(f"\n{'='*60}")
print(f"OVERALL: {total_pass}/{total_checks} test checks passed")
print('='*60)


---
## 📐 Scoring Rubric

| Test Check | Points | Description |
|-----------|--------|-------------|
| TC1 — Correct patient count | 5 | Exact match |
| TC2 — n_hvg = 100 | 5 | Must select exactly 100 HVGs |
| TC3 — PCs for 90% variance | 10 | Exact integer match |
| TC4 — Silhouette score | 15 | Within ±0.05 of expected |
| TC5 — ARI ≥ 0.9 | 20 | Clustering recovers true subtypes |
| TC6 — RF balanced accuracy ≥ 0.9 | 20 | Classifier performance |
| TC7 — Top DE genes (≥8/10 overlap) | 25 | Correct DE gene identification |
| **Total** | **100** | Per dataset |

**Passing threshold:** ≥70/100 per dataset, ≥80% across all three test sets.

---
## 🔑 Key Evaluation Criteria

- **Reproducibility:** Must use exact random seeds as specified
- **Pipeline correctness:** Each step must follow the specified algorithm choices
- **Numerical precision:** Floats reported to 4 decimal places
- **Output schema:** All required keys must be present in result dict
